# How much rank structure does each RDM share with the tree, and with each other?

**Question.** How well does the rank order of each RDM's distances agree with the a priori Kiani-Mur (KM) tree, and how do the RDMs agree with one another?

**Why it matters.** This is the most direct summary of "how similar is each RDM to the hierarchy". It makes no assumption about the shape of the relation (only monotonic order is compared), so it is the right first look before notebooks `3_` to `6_` ask where in the tree the agreement comes from.

**Statistics.** Kendall tau-a is primary for agreement with `sem_km`, which has only nine distinct values and therefore many ties (tau-a keeps ties in the denominator, so no model can reach 1). Spearman rho is secondary. Both are computed over the pairs the two RDMs both observe (`behav_joint` has about 13% unobserved pairs). Confidence intervals are 95% percentile intervals of an image bootstrap (images are the sampling unit, n_boot = 1000); tests against chance are image-relabeling permutation tests (n_perm = 1000, so the smallest possible p is 1/1001 = 0.001). Analytic p-values on RDM entries are never used, because the 262,450 entries come from 725 images and are not independent.

**RDMs.** `sem_km` (the reference), `sem_wn`, `clip_pre`, `clip_post`, `sens_pre`, `sens_post`, `behav_joint` (raw cohort-pooled behavior) and `behav_joint_mds_low` and `behav_joint_mds_high` (SMACOF embeddings of it at the two dimensionalities chosen in `1_mds_dimensionality` without looking at the tree: the stress elbow and the smallest d capturing 95% of the held-out prediction gain). Pre vs post (SHINE) is not compared here: that is `s1_pre_vs_post_shine/0_sensory`.

**Run from this directory.**

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parents[1]  # analysis/s2_semantic_hierarchy/ -> repo root
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import plotly.graph_objects as go

from analysis.s0_build_rdms.common import RESULTS_DIR
from analysis.s2_semantic_hierarchy import common as c
from analysis.s2_semantic_hierarchy import mds_dimensionality as md
from analysis.s2_semantic_hierarchy import rank_agreement as ra
from analysis.utils import plotting

if all((RESULTS_DIR / f"D_{n}.npy").exists() for n in c.BEHAV_MDS_SET):
    NAMES = c.FULL_SET
else:
    NAMES = c.RAW_SET
    print("NOTE: the MDS embeddings are not built yet (run 1_mds_dimensionality.ipynb): falling back to RAW_SET.")
RDMS = c.load_rdm_set(NAMES)
print(f"{len(NAMES)} RDMs: {', '.join(NAMES)}")
print(f"behav_joint: {np.isnan(RDMS[c.BEHAV]).mean():.1%} of pairs unobserved")

9 RDMs: sem_km, sem_wn, clip_pre, clip_post, sens_pre, sens_post, behav_joint, behav_joint_mds_low, behav_joint_mds_high
behav_joint: 13.1% of pairs unobserved


## 2.1 Do the RDMs agree with one another?

*Question:* which RDMs share rank structure, regardless of the tree? Why: a high agreement with the tree means more if the RDM is not simply a copy of another one, and the matrix shows which RDMs are redundant (for example pre and post versions).

In [2]:
for stat in c.RANK_STATS:
    corr = ra.correlation_matrix(RDMS, stat)
    ra.plot_correlation_matrix(corr, stat=stat).show()
    print(c.STAT_LABEL[stat])
    print(corr.round(2).to_string())

Kendall tau-a
                      sem_km  sem_wn  clip_pre  clip_post  sens_pre  sens_post  behav_joint  behav_joint_mds_low  behav_joint_mds_high
sem_km                  1.00    0.19      0.23       0.21      0.06       0.03         0.23                 0.37                  0.42
sem_wn                  0.19    1.00      0.03       0.02     -0.05       0.00         0.10                 0.12                  0.09
clip_pre                0.23    0.03      1.00       0.76      0.16       0.12         0.20                 0.33                  0.37
clip_post               0.21    0.02      0.76       1.00      0.16       0.12         0.19                 0.32                  0.36
sens_pre                0.06   -0.05      0.16       0.16      1.00       0.53         0.08                 0.13                  0.15
sens_post               0.03    0.00      0.12       0.12      0.53       1.00         0.08                 0.13                  0.13
behav_joint             0.23    0.10     

Spearman rho
                      sem_km  sem_wn  clip_pre  clip_post  sens_pre  sens_post  behav_joint  behav_joint_mds_low  behav_joint_mds_high
sem_km                  1.00    0.29      0.35       0.31      0.09       0.05         0.35                 0.56                  0.63
sem_wn                  0.29    1.00      0.05       0.04     -0.07       0.00         0.15                 0.18                  0.14
clip_pre                0.35    0.05      1.00       0.92      0.24       0.18         0.30                 0.47                  0.53
clip_post               0.31    0.04      0.92       1.00      0.24       0.18         0.28                 0.46                  0.51
sens_pre                0.09   -0.07      0.24       0.24      1.00       0.73         0.11                 0.19                  0.22
sens_post               0.05    0.00      0.18       0.18      0.73       1.00         0.12                 0.19                  0.20
behav_joint             0.35    0.15      

Pre and post versions of the same RDM are tightly related (CLIP pre/post: tau-a 0.76, rho 0.92; pixels pre/post: tau-a 0.53, rho 0.73). Across families agreement is low: WordNet is almost unrelated to CLIP (tau-a 0.03) and to pixels (-0.05), whereas `sem_km` shares 0.19 to 0.23 with WordNet, CLIP and behavior. Behavior sits closest to the tree and to CLIP (tau-a 0.23 and 0.20) and farther from WordNet (0.10) and pixels (0.08). Pixel RDMs are almost unrelated to the tree (0.06, 0.03).

## 2.2 How well does each RDM agree with the tree?

*Question:* the point estimate, its uncertainty and its distance from chance, for every RDM against `sem_km`. Why: this is the central quantity of the notebook. Table columns: `_lo`/`_hi` are the bootstrap 95% CI, `_p` the label-shuffle p (floor 0.001).

In [3]:
TABLE = ra.agreement_with_reference(RDMS, n_boot=ra.N_BOOT, n_perm=ra.N_PERM)
shown = TABLE.copy()
shown.index = [plotting.rdm_label(n) for n in shown.index]
print(shown.round(3).to_string())
ra.plot_forest(TABLE, stat="kendall_a").show()
ra.plot_forest(TABLE, stat="spearman").show()

                            n_pairs  kendall_a  kendall_a_lo  kendall_a_hi  kendall_a_p  spearman  spearman_lo  spearman_hi  spearman_p
WordNet                      262450      0.189         0.155         0.222        0.001     0.286        0.239        0.333       0.001
CLIP pre                     262450      0.233         0.210         0.260        0.001     0.348        0.314        0.386       0.001
CLIP post                    262450      0.207         0.184         0.234        0.001     0.310        0.277        0.349       0.001
pixels pre                   262450      0.058         0.035         0.084        0.001     0.090        0.054        0.129       0.001
pixels post                  262450      0.034         0.013         0.057        0.001     0.054        0.021        0.089       0.001
behavior joint               228001      0.231         0.221         0.243        0.001     0.351        0.336        0.368       0.001
behavior joint, MDS low-d    262450      0.372  

In [4]:
for name in c.BEHAV_MDS_SET:
    if name not in NAMES:
        print(f"{name} not available in this run (RAW_SET).")
        continue
    row, raw = TABLE.loc[name], TABLE.loc[c.BEHAV]
    print(f"{plotting.rdm_label(name)} vs raw behavior: tau-a {row['kendall_a']:.3f} [{row['kendall_a_lo']:.3f}, {row['kendall_a_hi']:.3f}] "
          f"vs {raw['kendall_a']:.3f}; Spearman {row['spearman']:.3f} vs {raw['spearman']:.3f}")

behavior joint, MDS low-d vs raw behavior: tau-a 0.372 [0.354, 0.391] vs 0.231; Spearman 0.557 vs 0.351
behavior joint, MDS high-d vs raw behavior: tau-a 0.421 [0.401, 0.439] vs 0.231; Spearman 0.625 vs 0.351


Every RDM agrees with the tree above chance (all p = 0.001, the permutation floor). CLIP pre and raw behavior are the closest to the tree (tau-a 0.233 and 0.231; rho 0.348 and 0.351); CLIP post is slightly lower (0.207), WordNet lower still (0.189, rho 0.286), and the pixel RDMs are near zero (0.058 and 0.034), with CIs that exclude 0 but are very small. Behavior's interval is the tightest (tau-a 0.221 to 0.243). Nothing here approaches 1: even the best RDMs recover a modest fraction of the tree's rank order, partly because the nine-valued tree caps tau-a below 1.

## 2.3 Behavioral RDMs against their ceiling

*Question:* how much of the tree could a perfect model of the *behavioral signal* explain? Why: a behavioral RDM is noisy, so its agreement with any fixed target is capped. Judging 0.35 against 1 would understate it.

The bound: the raw joint RDM's split-half Spearman, projected to the full sample with the Spearman-Brown formula (r_SB), gives the highest Spearman any noise-free model could reach with it, sqrt(r_SB). We report each behavioral RDM's Spearman with KM as a fraction of this bound. Two caveats: (i) the bound is derived for the **raw** RDM's reliability, so it does **not** bound the two MDS embeddings: they are smoothed models of the data whose noise is partly averaged out, and their Spearman with KM may exceed sqrt(r_SB) (the fraction is then above 1 and only indicative); (ii) KM's ties cap tau-a (and, less severely, Spearman) below 1 for any RDM, so the fraction is a ceiling for noise, not for ties, and tau-a is not rescaled.

In [5]:
trials, n_images = md.joint_trials()
ceil = md.raw_ceiling(trials, n_images)
R_SB = float(ceil["spearman_brown"].mean())
BOUND = md.attenuation_bound(R_SB)
print(f"raw joint RDM split-half Spearman: {ceil['spearman'].mean():.3f}; Spearman-Brown r_SB = {R_SB:.3f}; "
      f"bound sqrt(r_SB) = {BOUND:.3f}")
behav_names = [n for n in (c.BEHAV, *c.BEHAV_MDS_SET) if n in NAMES]
CEIL = ra.ceiling_table(TABLE, R_SB, behav_names)
CEIL.index = [plotting.rdm_label(n) for n in CEIL.index]
print(CEIL.round(3).to_string())

raw joint RDM split-half Spearman: 0.220; Spearman-Brown r_SB = 0.361; bound sqrt(r_SB) = 0.601
                            spearman  bound  fraction_of_bound  kendall_a
behavior joint                 0.351  0.601              0.583      0.231
behavior joint, MDS low-d      0.557  0.601              0.928      0.372
behavior joint, MDS high-d     0.625  0.601              1.041      0.421


The raw joint RDM has a split-half Spearman of 0.220, which projects to r_SB = 0.361 and a bound of sqrt(r_SB) = 0.601. Raw behavior's Spearman with KM (0.351) is therefore 58% of what a noise-free model could reach. The remainder is not "unexplained by semantics": it is behavioral signal the tree does not carry, plus the coarseness of KM itself.

## 2.4 Which differences are real?

*Question:* the old notebook said "CLIP tracks the tree better than WordNet" (rho 0.35 vs 0.29) without a test. Here only the rankings the takeaways state are tested, with a **paired image bootstrap**: each draw resamples images once and computes both RDMs' agreement with KM on those same images and on the pairs all three RDMs observe; the CI is that of the difference. p = 2 min(P(diff <= 0), P(diff >= 0)), floored at 1/1000, then Benjamini-Hochberg FDR over this small family (on tau-a). The comparison of behavior with "the best reference RDM" uses the reference with the highest tau-a in 2.2, which is chosen from the same data. Pre vs post is not tested here (that is SHINE, s1).

In [6]:
references = [n for n in NAMES if n not in (c.REFERENCE, c.BEHAV, *c.BEHAV_MDS_SET)]
BEST = TABLE.loc[references, "kendall_a"].idxmax()
print(f"best-agreeing reference RDM (tau-a with KM): {plotting.rdm_label(BEST)}")
PAIRS = [("clip_pre", "sem_wn")] + [(b, BEST) for b in (c.BEHAV, *c.BEHAV_MDS_SET) if b in NAMES]
DIFF = ra.paired_family(RDMS, PAIRS)
shown = DIFF.copy()
shown.index = [f"{plotting.rdm_label(a)} - {plotting.rdm_label(b)}" for a, b in shown.index]
print(shown.round(4).to_string())

best-agreeing reference RDM (tau-a with KM): CLIP pre


                                       n_pairs  kendall_a  kendall_a_lo  kendall_a_hi  kendall_a_p  spearman  spearman_lo  spearman_hi  spearman_p  p_fdr
CLIP pre - WordNet                      262450     0.0448       -0.0009        0.0921        0.054    0.0625      -0.0017       0.1289       0.060  0.072
behavior joint - CLIP pre               228001    -0.0031       -0.0252        0.0180        0.804    0.0008      -0.0317       0.0311       0.960  0.804
behavior joint, MDS low-d - CLIP pre    262450     0.1384        0.1120        0.1637        0.001    0.2093       0.1714       0.2445       0.001  0.002
behavior joint, MDS high-d - CLIP pre   262450     0.1872        0.1611        0.2142        0.001    0.2772       0.2402       0.3129       0.001  0.002


CLIP pre minus WordNet: tau-a +0.045 (95% CI -0.001 to 0.092, p = 0.054; rho +0.062, CI -0.002 to 0.129). The old claim that CLIP tracks the tree better than WordNet is **not supported**: the difference is in the expected direction but its interval includes 0 (n_pairs = 262,450). Raw behavior minus CLIP pre: tau-a -0.003 (CI -0.025 to 0.018, p = 0.80; rho +0.001, CI -0.032 to 0.031): behavior and the best stimulus-derived RDM are indistinguishable in how well they track the tree. No comparison survives FDR correction (p_fdr column).

## 2.5 Sensitivity: identical pairs for every RDM

*Question:* is the comparison of behavior with the other RDMs distorted by the unobserved pairs? The point estimates of 2.2 are recomputed with every RDM restricted to the pairs `behav_joint` observed.

In [7]:
SENS = ra.agreement_on_pairs_of(RDMS, c.BEHAV)
SENS["full_data_kendall_a"] = TABLE["kendall_a"]
SENS["full_data_spearman"] = TABLE["spearman"]
SENS.index = [plotting.rdm_label(n) for n in SENS.index]
print(f"All RDMs restricted to the {int(SENS['n_pairs'].iloc[0]):,} pairs behav_joint observed:")
print(SENS.round(3).to_string())

All RDMs restricted to the 228,001 pairs behav_joint observed:
                            n_pairs  kendall_a  spearman  full_data_kendall_a  full_data_spearman
WordNet                      228001      0.187     0.284                0.189               0.286
CLIP pre                     228001      0.234     0.350                0.233               0.348
CLIP post                    228001      0.208     0.312                0.207               0.310
pixels pre                   228001      0.059     0.091                0.058               0.090
pixels post                  228001      0.035     0.055                0.034               0.054
behavior joint               228001      0.231     0.351                0.231               0.351
behavior joint, MDS low-d    228001      0.373     0.559                0.372               0.557
behavior joint, MDS high-d   228001      0.421     0.626                0.421               0.625


Restricting to the 228,001 pairs behavior observed changes nothing at the second decimal for the references (e.g. CLIP pre tau-a 0.234 vs 0.233, WordNet 0.187 vs 0.189), so the unobserved pairs do not bias the comparison between behavior and the reference RDMs.

## Takeaways

* All RDMs agree with the KM tree above chance (every label-shuffle p = 0.001, the permutation floor), but among the raw RDMs only modestly: the best tau-a values are 0.233 (CLIP pre) and 0.231 (raw behavior), Spearman 0.348 and 0.351.
* Ranking of the raw RDMs by tau-a: CLIP pre 0.233 ~ behavior 0.231 > CLIP post 0.207 > WordNet 0.189 >> pixels (0.058 pre, 0.034 post). Pixels share almost no rank structure with the tree.
* **Embedding the behavioral RDM raises its agreement with the tree by 60 to 80%.** tau-a rises from 0.231 (raw) to 0.372 [0.354, 0.391] for the low-d embedding (d = 5) and 0.421 [0.401, 0.439] for the high-d one (d = 20); Spearman from 0.351 to 0.557 and 0.625. Both embeddings exceed CLIP pre, the best-agreeing reference RDM (tau-a +0.138 [0.112, 0.164] and +0.187 [0.161, 0.214], p_fdr = 0.002 for both).
* **Raw behavior reaches 58% of its reliability ceiling** (Spearman 0.351 of sqrt(r_SB) = 0.601, r_SB = 0.361). The embeddings reach 0.93 and 1.04 of that value, which shows that the ceiling does not apply to them: they are smoothed models of the ratings, not noisy measurements with the raw RDM's noise. Their higher agreement says the smoothing discards rating noise that does not follow the tree, not that behavior is more reliable than measured.
* **The old claim "CLIP tracks the tree better than WordNet" is not supported:** tau-a difference +0.045, 95% CI -0.001 to 0.092, p = 0.054, p_fdr = 0.072 over the four tested differences (rho +0.062, CI -0.002 to 0.129). Raw behavior does not differ detectably from CLIP pre (tau-a -0.003, CI -0.025 to 0.018, p_fdr = 0.80).
* **The RDM families share little with each other.** WordNet is nearly orthogonal to CLIP (tau-a 0.03) and pixels (-0.05), and weakly related to behavior (0.10 raw); CLIP and behavior share more (0.20 raw, 0.33 to 0.37 embedded). Each pre/post pair agrees strongly (CLIP 0.76, pixels 0.53), and the two embeddings agree with each other at 0.68.
* The unobserved 13% of behavioral pairs do not distort the comparison: restricted to identical pairs, every RDM's agreement changes by at most 0.002.
* Limitations: (a) KM has nine distinct values, so tau-a is capped below 1 and absolute values are not comparable to a noise-free ceiling; (b) agreement is one global number that cannot say where in the tree it comes from (notebooks 3 and 4), nor whether pixel similarity explains it (5); (c) the CIs treat images as exchangeable, ignoring that images of the same category are related; (d) the "best reference" comparison is chosen from the same data.